In [26]:
# Standard library imports
import sys
from pathlib import Path

# Third-party imports
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

# Display settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
%matplotlib inline

## 1. Initialize Data Connection (DuckDB)

**Optimization:** Using DuckDB to query the Parquet file directly.

In [27]:
# Define path to dataset
dashboard_path = Path('../data/processed/dashboard_data.parquet')

if not dashboard_path.exists():
    raise FileNotFoundError(
        "Dashboard dataset not found! Please run notebook 06_feature_engineering.ipynb first."
    )

print("Initializing DuckDB connection...")
con = duckdb.connect(database=':memory:')

# Create a view
con.execute(f"CREATE OR REPLACE VIEW crashes AS SELECT * FROM '{dashboard_path}'")

# Inspect columns to find the correct date column name
columns_info = con.execute("DESCRIBE crashes").fetchall()
columns = [col[0] for col in columns_info]
print(f"Available columns: {columns}")

# Helper to find column case-insensitively
def find_col(candidates):
    for cand in candidates:
        if cand in columns:
            return f'"{cand}"'
    return None

date_col = find_col(['CRASH DATE', 'CRASH_DATE', 'crash_date'])
if not date_col:
    raise ValueError("Could not find crash date column!")

print(f"Using date column: {date_col}")

# Get basic stats
total_records = con.execute("SELECT COUNT(*) FROM crashes").fetchone()[0]
date_range = con.execute(f"SELECT MIN({date_col}), MAX({date_col}) FROM crashes").fetchone()

print(f"✓ Connected to dataset")
print(f"✓ Total Records: {total_records:,}")
print(f"✓ Date Range: {date_range[0]} to {date_range[1]}")

Initializing DuckDB connection...
Available columns: ['COLLISION_ID', 'PERSON_TYPE', 'PERSON_AGE', 'PERSON_SEX', 'PERSON_INJURY', 'PED_ROLE', 'COMPLAINT', 'BODILY_INJURY', 'POSITION_IN_VEHICLE', 'CRASH DATE', 'CRASH TIME', 'BOROUGH', 'ZIP CODE', 'LATITUDE', 'LONGITUDE', 'CONTRIBUTING FACTOR VEHICLE 1', 'VEHICLE TYPE CODE 1', 'crash_year', 'crash_month', 'crash_day', 'crash_day_of_week', 'crash_day_name', 'crash_quarter', 'crash_week_of_year', 'crash_time_obj', 'crash_hour', 'season', 'borough_clean', 'has_borough', 'has_coordinates', 'severity_score']
Using date column: "CRASH DATE"
✓ Connected to dataset
✓ Total Records: 5,377,418
✓ Date Range: 2012-07-02 00:00:00 to 2025-11-04 00:00:00
✓ Connected to dataset
✓ Total Records: 5,377,418
✓ Date Range: 2012-07-02 00:00:00 to 2025-11-04 00:00:00


## 2. Temporal Analysis

Analyze crash trends over time.

### Research Question 1: How have crash frequencies changed over time?

**Hypothesis:** We expect to see a decline in crashes over recent years due to improved traffic safety measures and vehicle technology advancements.

In [28]:
# 2.1 Yearly Trend
print("Analyzing yearly crash trends...")
query_year = """
    SELECT crash_year, COUNT(*) as crashes 
    FROM crashes 
    WHERE crash_year IS NOT NULL
    GROUP BY 1 
    ORDER BY 1
"""
df_year = con.execute(query_year).df()

fig_year = px.line(df_year, x='crash_year', y='crashes', 
                   title='Total Crashes per Year',
                   markers=True)
fig_year.show()

print(f"**Interpretation:** The data shows {df_year['crashes'].iloc[0]:,} crashes in {df_year['crash_year'].iloc[0]} "
      f"and {df_year['crashes'].iloc[-1]:,} crashes in {df_year['crash_year'].iloc[-1]}. "
      f"{'A declining trend suggests improved road safety measures.' if df_year['crashes'].iloc[-1] < df_year['crashes'].iloc[0] else 'An increasing trend may indicate growing traffic volume or reporting improvements.'}")

Analyzing yearly crash trends...


**Interpretation:** The data shows 225 crashes in 2012 and 236,451 crashes in 2025. An increasing trend may indicate growing traffic volume or reporting improvements.


### Research Question 2: What are the peak hours for motor vehicle collisions?

**Hypothesis:** Rush hours (8-9 AM and 5-6 PM) will show the highest crash frequencies due to increased traffic volume and congestion.

In [29]:
# 2.2 Hourly Trend
print("Analyzing hourly crash patterns...")
query_hour = """
    SELECT crash_hour, COUNT(*) as crashes 
    FROM crashes 
    WHERE crash_hour IS NOT NULL
    GROUP BY 1 
    ORDER BY 1
"""
df_hour = con.execute(query_hour).df()

fig_hour = px.bar(df_hour, x='crash_hour', y='crashes', 
                  title='Crashes by Hour of Day',
                  color='crashes')
fig_hour.show()

peak_hour = df_hour.loc[df_hour['crashes'].idxmax()]
print(f"**Interpretation:** Peak crash hour is {int(peak_hour['crash_hour'])}:00 with {peak_hour['crashes']:,.0f} crashes. "
      f"Evening rush hours show elevated crash rates, likely due to driver fatigue combined with high traffic volume.")

Analyzing hourly crash patterns...


**Interpretation:** Peak crash hour is 16:00 with 388,540 crashes. Evening rush hours show elevated crash rates, likely due to driver fatigue combined with high traffic volume.


## 3. Location Analysis

Analyze crashes by Borough.

### Research Question 3: Which NYC boroughs experience the most crashes?

**Hypothesis:** Brooklyn and Queens will have the highest crash counts due to their larger populations and road network density.

In [30]:
# 3.1 Crashes by Borough
print("Analyzing crashes by borough...")
query_borough = """
    SELECT borough_clean, COUNT(*) as crashes 
    FROM crashes 
    WHERE borough_clean IS NOT NULL
    GROUP BY 1 
    ORDER BY 2 DESC
"""
df_borough = con.execute(query_borough).df()

fig_borough = px.bar(df_borough, x='borough_clean', y='crashes', 
                     title='Total Crashes by Borough',
                     color='borough_clean')
fig_borough.show()

top_borough = df_borough.iloc[0]
print(f"**Interpretation:** {top_borough['borough_clean']} has the highest crash count with {top_borough['crashes']:,.0f} crashes, "
      f"representing {100 * top_borough['crashes'] / df_borough['crashes'].sum():.1f}% of all crashes. "
      f"This correlates with higher population density and traffic volume in these areas.")

Analyzing crashes by borough...


**Interpretation:** Brooklyn has the highest crash count with 3,122,340 crashes, representing 58.1% of all crashes. This correlates with higher population density and traffic volume in these areas.


### Research Question 4: Do crash patterns vary by day of the week?

**Hypothesis:** Weekdays will show higher crash frequencies than weekends due to commuter traffic, with Friday potentially showing elevated rates as drivers are more fatigued at the end of the work week.

In [31]:
# 3.2 Crashes by Day of Week
print("Analyzing crash patterns by day of week...")

# Find day of week column
day_name_col = find_col(['crash_day_name', 'CRASH_DAY_NAME', 'day_of_week'])
day_col = find_col(['crash_day_of_week', 'CRASH_DAY_OF_WEEK'])

if day_name_col and day_col:
    query_day = f"""
        SELECT 
            {day_col} as day_num,
            {day_name_col} as day_name, 
            COUNT(*) as crashes,
            AVG(severity_score) as avg_severity
        FROM crashes 
        WHERE {day_name_col} IS NOT NULL
        GROUP BY 1, 2
        ORDER BY 1
    """
    df_day = con.execute(query_day).df()
    
    # Create a grouped bar chart showing crashes and severity
    fig_day = go.Figure()
    
    # Add crashes bar
    fig_day.add_trace(go.Bar(
        x=df_day['day_name'],
        y=df_day['crashes'],
        name='Total Crashes',
        marker_color='steelblue',
        yaxis='y'
    ))
    
    # Add average severity line
    fig_day.add_trace(go.Scatter(
        x=df_day['day_name'],
        y=df_day['avg_severity'],
        name='Avg Severity Score',
        marker_color='red',
        yaxis='y2',
        mode='lines+markers'
    ))
    
    # Update layout with dual y-axes
    fig_day.update_layout(
        title='Crash Frequency and Severity by Day of Week',
        xaxis=dict(title='Day of Week'),
        yaxis=dict(title='Number of Crashes', side='left'),
        yaxis2=dict(title='Average Severity Score', side='right', overlaying='y'),
        legend=dict(x=0.01, y=0.99),
        height=500
    )
    
    fig_day.show()
    
    # Find peak day
    peak_day = df_day.loc[df_day['crashes'].idxmax()]
    lowest_day = df_day.loc[df_day['crashes'].idxmin()]
    
    print(f"\n**Interpretation:** {peak_day['day_name']} has the highest crash count with {peak_day['crashes']:,.0f} crashes, "
          f"while {lowest_day['day_name']} has the lowest with {lowest_day['crashes']:,.0f} crashes. ")
    
    # Calculate weekday vs weekend
    weekday_crashes = df_day[df_day['day_num'] < 5]['crashes'].sum()
    weekend_crashes = df_day[df_day['day_num'] >= 5]['crashes'].sum()
    
    print(f"Weekdays account for {weekday_crashes:,.0f} crashes ({100*weekday_crashes/(weekday_crashes+weekend_crashes):.1f}%), "
          f"while weekends account for {weekend_crashes:,.0f} crashes ({100*weekend_crashes/(weekday_crashes+weekend_crashes):.1f}%). "
          f"This pattern reflects the higher traffic volume during workdays versus weekends.")
else:
    print("Day of week columns not found.")

Analyzing crash patterns by day of week...



**Interpretation:** Friday has the highest crash count with 855,398 crashes, while Sunday has the lowest with 672,289 crashes. 
Weekdays account for 3,771,410 crashes (70.1%), while weekends account for 1,606,006 crashes (29.9%). This pattern reflects the higher traffic volume during workdays versus weekends.


### Research Question 5: Where are the geographic hotspots for high-severity crashes?

**Hypothesis:** High-severity crashes will cluster along major highways and arterial roads, with concentrations in areas with high traffic volume and complex intersections.

In [32]:
# 3.3 High Severity Map (Top 1000)
print("Mapping high severity crashes...")
# We need to find the correct latitude/longitude columns too
lat_col = find_col(['LATITUDE', 'latitude'])
lon_col = find_col(['LONGITUDE', 'longitude'])

if lat_col and lon_col:
    # Check what columns are available for calculating severity
    casualties_col = find_col(['total_casualties', 'TOTAL_CASUALTIES'])
    severity_cat_col = find_col(['severity_category', 'SEVERITY_CATEGORY'])
    
    # Find injury/death columns to calculate casualties if needed
    if not casualties_col:
        injured_cols = []
        killed_cols = []
        for prefix in ['NUMBER OF PEDESTRIANS', 'NUMBER OF CYCLIST', 'NUMBER OF MOTORIST',
                       'NUMBER_OF_PEDESTRIANS', 'NUMBER_OF_CYCLIST', 'NUMBER_OF_MOTORIST']:
            injured = find_col([f'{prefix} INJURED', f'{prefix}_INJURED'])
            killed = find_col([f'{prefix} KILLED', f'{prefix}_KILLED'])
            if injured:
                injured_cols.append(injured)
            if killed:
                killed_cols.append(killed)
        
        # Build casualties calculation
        if injured_cols or killed_cols:
            casualties_expr = " + ".join(injured_cols + killed_cols)
            casualties_calc = f"({casualties_expr}) as total_casualties"
        else:
            casualties_calc = "1 as total_casualties"  # Fallback
    else:
        casualties_calc = f"{casualties_col} as total_casualties"
    
    # Build query based on available columns
    if severity_cat_col:
        query_map = f"""
            SELECT {lat_col} as lat, {lon_col} as lon, {casualties_calc}, borough_clean
            FROM crashes 
            WHERE has_coordinates = TRUE 
              AND {severity_cat_col} = 'High'
            ORDER BY total_casualties DESC
            LIMIT 1000
        """
    else:
        # Fallback: order by calculated casualties
        query_map = f"""
            SELECT {lat_col} as lat, {lon_col} as lon, {casualties_calc}, borough_clean
            FROM crashes 
            WHERE has_coordinates = TRUE 
              AND {casualties_calc.split(' as ')[0]} >= 3
            ORDER BY total_casualties DESC
            LIMIT 1000
        """
    
    df_map = con.execute(query_map).df()

    if len(df_map) > 0:
        fig_map = px.scatter_mapbox(df_map, lat="lat", lon="lon", 
                                    color="borough_clean", size="total_casualties",
                                    zoom=10, height=600,
                                    title="Top 1000 High Severity Crashes")
        fig_map.update_layout(mapbox_style="open-street-map")
        fig_map.show()
        
        print(f"**Interpretation:** The map reveals {len(df_map)} high-severity crash hotspots. "
              f"Geographic clustering indicates specific high-risk corridors that warrant targeted safety interventions.")
    else:
        print("No high-severity crashes found with coordinates.")
else:
    print("Latitude/Longitude columns not found for mapping.")


Mapping high severity crashes...
No high-severity crashes found with coordinates.


## 4. Severity & Contributing Factors

Analyze what causes crashes and how severe they are.

### Research Question 6: What is the distribution of crash severity?

**Hypothesis:** Most crashes will be low severity (property damage only), with fewer moderate and high severity incidents.

In [34]:
# 4.1 Severity Distribution
print("Analyzing crash severity distribution...")

# Create severity categories based on severity_score
query_sev = """
    SELECT 
        CASE 
            WHEN severity_score = 0 THEN 'No Injury'
            WHEN severity_score > 0 AND severity_score <= 2 THEN 'Low'
            WHEN severity_score > 2 AND severity_score <= 5 THEN 'Moderate'
            ELSE 'High'
        END as severity_category,
        COUNT(*) as crashes 
    FROM crashes 
    WHERE severity_score IS NOT NULL
    GROUP BY 1
    ORDER BY 2 DESC
"""
df_sev = con.execute(query_sev).df()

fig_sev = px.pie(df_sev, values='crashes', names='severity_category', 
                 title='Distribution of Crash Severity')
fig_sev.show()

for idx, row in df_sev.iterrows():
    pct = 100 * row['crashes'] / df_sev['crashes'].sum()
    print(f"{row['severity_category']}: {row['crashes']:,.0f} crashes ({pct:.1f}%)")
    
print(f"\n**Interpretation:** The severity distribution confirms that most crashes result in minimal casualties, "
      f"but the presence of high-severity crashes underscores the need for continued safety improvements.")

Analyzing crash severity distribution...


No Injury: 4,844,462 crashes (90.1%)
Low: 529,450 crashes (9.8%)
High: 3,506 crashes (0.1%)

**Interpretation:** The severity distribution confirms that most crashes result in minimal casualties, but the presence of high-severity crashes underscores the need for continued safety improvements.


### Research Question 7: What are the most common contributing factors to crashes?

**Hypothesis:** Driver inattention/distraction will be the leading contributing factor, followed by failure to yield and following too closely.

In [35]:
# 4.2 Top Contributing Factors
print("Analyzing contributing factors...")
# Find the correct column for contributing factor
factor_col = find_col(['CONTRIBUTING FACTOR VEHICLE 1', 'CONTRIBUTING_FACTOR_VEHICLE_1'])

if factor_col:
    query_factor = f"""
        SELECT {factor_col} as factor, COUNT(*) as crashes 
        FROM crashes 
        WHERE {factor_col} IS NOT NULL 
          AND {factor_col} != 'Unspecified'
        GROUP BY 1 
        ORDER BY 2 DESC 
        LIMIT 10
    """
    df_factor = con.execute(query_factor).df()

    fig_factor = px.bar(df_factor, y='factor', x='crashes', 
                        title='Top 10 Contributing Factors',
                        orientation='h')
    fig_factor.update_layout(yaxis={'categoryorder':'total ascending'})
    fig_factor.show()
    
    top_factor = df_factor.iloc[0]
    print(f"\n**Interpretation:** '{top_factor['factor']}' is the leading cause with {top_factor['crashes']:,.0f} crashes. "
          f"These behavioral factors suggest that driver education and enforcement could significantly reduce crash rates.")
else:
    print("Contributing factor column not found.")

Analyzing contributing factors...



**Interpretation:** 'Driver Inattention/Distraction' is the leading cause with 1,346,736 crashes. These behavioral factors suggest that driver education and enforcement could significantly reduce crash rates.


## 5. Road User Analysis

### Research Question 8: How do injuries differ across road user types (pedestrians, cyclists, motorists) by borough?

**Hypothesis:** Pedestrian injuries will be highest in Manhattan due to high foot traffic, while motorist injuries will dominate in outer boroughs with more vehicular traffic.

In [37]:
# 5.1 Road User Injuries by Borough
print("Analyzing road user injuries by borough...")

# Since this is person-level data, we need to count injuries by person type and borough
query_road_users = """
    SELECT 
        borough_clean,
        COUNT(*) FILTER (WHERE PERSON_TYPE = 'Pedestrian' AND PERSON_INJURY = 'Injured') as pedestrian_injuries,
        COUNT(*) FILTER (WHERE PERSON_TYPE = 'Bicyclist' AND PERSON_INJURY = 'Injured') as cyclist_injuries,
        COUNT(*) FILTER (WHERE PERSON_TYPE IN ('Driver', 'Occupant') AND PERSON_INJURY = 'Injured') as motorist_injuries
    FROM crashes
    WHERE borough_clean IS NOT NULL
    GROUP BY 1
    ORDER BY 1
"""
df_users = con.execute(query_road_users).df()

# Melt for grouped bar chart
df_melted = df_users.melt(id_vars='borough_clean', 
                           value_vars=['pedestrian_injuries', 'cyclist_injuries', 'motorist_injuries'],
                           var_name='user_type', value_name='injuries')

fig_users = px.bar(df_melted, x='borough_clean', y='injuries', 
                   color='user_type', barmode='group',
                   title='Road User Injuries by Borough',
                   labels={'injuries': 'Total Injuries', 'borough_clean': 'Borough'})
fig_users.show()

total_injuries = df_users[['pedestrian_injuries', 'cyclist_injuries', 'motorist_injuries']].sum()
print(f"\n**Interpretation:** Across all boroughs, motorists account for {total_injuries['motorist_injuries']:,.0f} injuries, "
      f"pedestrians {total_injuries['pedestrian_injuries']:,.0f}, and cyclists {total_injuries['cyclist_injuries']:,.0f}. "
      f"The distribution varies by borough, reflecting differences in transportation modes and infrastructure.")

Analyzing road user injuries by borough...



**Interpretation:** Across all boroughs, motorists account for 380,164 injuries, pedestrians 89,704, and cyclists 49,597. The distribution varies by borough, reflecting differences in transportation modes and infrastructure.


## 6. Vehicle Analysis

### Research Question 9: What types of vehicles are most frequently involved in crashes?

**Hypothesis:** Passenger vehicles (sedans, SUVs) will dominate crash involvement due to their prevalence on NYC roads, with commercial vehicles also showing significant presence.

In [38]:
# 6.1 Vehicle Types Involved in Crashes
print("Analyzing vehicle types...")

# Find vehicle type column
veh_type_col = find_col(['VEHICLE TYPE CODE 1', 'VEHICLE_TYPE_CODE_1', 'vehicle_type_code_1'])

if veh_type_col:
    query_vehicle = f"""
        SELECT {veh_type_col} as vehicle_type, COUNT(*) as crashes 
        FROM crashes 
        WHERE {veh_type_col} IS NOT NULL 
          AND {veh_type_col} != 'UNKNOWN'
        GROUP BY 1 
        ORDER BY 2 DESC 
        LIMIT 10
    """
    df_vehicle = con.execute(query_vehicle).df()

    fig_vehicle = px.bar(df_vehicle, y='vehicle_type', x='crashes', 
                        title='Top 10 Vehicle Types Involved in Crashes',
                        orientation='h',
                        color='crashes')
    fig_vehicle.update_layout(yaxis={'categoryorder':'total ascending'})
    fig_vehicle.show()
    
    top_vehicle = df_vehicle.iloc[0]
    total_top10 = df_vehicle['crashes'].sum()
    print(f"\n**Interpretation:** '{top_vehicle['vehicle_type']}' is involved in {top_vehicle['crashes']:,.0f} crashes. "
          f"The top 10 vehicle types account for {total_top10:,.0f} crashes, "
          f"highlighting which vehicle categories should be prioritized for safety interventions.")
else:
    print("Vehicle type column not found.")

Analyzing vehicle types...



**Interpretation:** 'Sedan' is involved in 2,383,628 crashes. The top 10 vehicle types account for 5,151,273 crashes, highlighting which vehicle categories should be prioritized for safety interventions.


## 7. Time-Based Severity Analysis

### Research Question 10: Does crash severity vary by time of day?

**Hypothesis:** Late night/early morning hours (midnight-6 AM) will show higher average severity due to factors like impaired driving, reduced visibility, and higher speeds on less congested roads.

In [39]:
# 7.1 Average Severity by Hour
print("Analyzing severity patterns by time of day...")

# Find severity score column
severity_col = find_col(['severity_score', 'SEVERITY_SCORE'])

if severity_col:
    query_severity_hour = f"""
        SELECT 
            crash_hour,
            AVG({severity_col}) as avg_severity,
            COUNT(*) as crash_count
        FROM crashes 
        WHERE crash_hour IS NOT NULL
          AND {severity_col} IS NOT NULL
        GROUP BY 1 
        ORDER BY 1
    """
    df_sev_hour = con.execute(query_severity_hour).df()

    fig_sev_hour = px.line(df_sev_hour, x='crash_hour', y='avg_severity', 
                           title='Average Crash Severity by Hour of Day',
                           markers=True,
                           labels={'avg_severity': 'Average Severity Score', 'crash_hour': 'Hour'})
    fig_sev_hour.show()
    
    max_severity_hour = df_sev_hour.loc[df_sev_hour['avg_severity'].idxmax()]
    min_severity_hour = df_sev_hour.loc[df_sev_hour['avg_severity'].idxmin()]
    print(f"\n**Interpretation:** Peak severity occurs at {int(max_severity_hour['crash_hour'])}:00 "
          f"(avg severity: {max_severity_hour['avg_severity']:.3f}), while lowest severity is at "
          f"{int(min_severity_hour['crash_hour'])}:00 (avg severity: {min_severity_hour['avg_severity']:.3f}). "
          f"This temporal pattern suggests that crash characteristics differ significantly throughout the day, "
          f"with nighttime crashes tending to be more severe despite lower frequency.")
else:
    print("Severity score column not found.")

Analyzing severity patterns by time of day...



**Interpretation:** Peak severity occurs at 4:00 (avg severity: 0.152), while lowest severity is at 10:00 (avg severity: 0.085). This temporal pattern suggests that crash characteristics differ significantly throughout the day, with nighttime crashes tending to be more severe despite lower frequency.


In [ ]:
# Note: Connection will be closed after validation
print("Analysis complete. Running validation next...")

Analysis complete.


## Data Validation: Injury/Death Statistics

**Objective:** Validate injury and death counts against NYC official statistics to ensure data quality.

**Expected Ranges (2012-2025):**
- **Deaths**: ~3,000-3,500 (based on NYC Vision Zero reports, averaging 200-250 fatalities per year)
- **Injuries**: ~500,000-700,000 (based on NYC DOT collision reports, averaging 40,000-60,000 injuries per year)

**Sources:**
- NYC Vision Zero: https://www.nyc.gov/content/visionzero/pages/
- NYC OpenData Motor Vehicle Collisions: https://data.cityofnewyork.us/
- NYPD Traffic Safety: https://www.nyc.gov/site/nypd/stats/traffic-data/traffic-data.page

In [ ]:
# Query total injuries and deaths from person-level data
person_injury_col = None
for col in ['PERSON_INJURY', 'person_injury']:
    if col in columns:
        person_injury_col = f'"{col}"'
        break

if person_injury_col:
    validation_query = f"""
    SELECT 
        '{date_range[0]}'::DATE as from_date,
        '{date_range[1]}'::DATE as to_date,
        COUNT(DISTINCT COLLISION_ID) as total_crashes,
        COUNT(*) FILTER (WHERE {person_injury_col} = 'Injured') as total_injuries,
        COUNT(*) FILTER (WHERE {person_injury_col} = 'Killed') as total_deaths,
        (COUNT(*) FILTER (WHERE {person_injury_col} = 'Injured') + 
         COUNT(*) FILTER (WHERE {person_injury_col} = 'Killed')) as total_casualties
    FROM crashes
    """
    
    df_validation = con.execute(validation_query).fetchdf()
    
    print("=" * 80)
    print("DATA VALIDATION RESULTS")
    print("=" * 80)
    print(f"\n📅 Date Range: {df_validation['from_date'][0]} to {df_validation['to_date'][0]}")
    print(f"\n🚗 Total Crashes: {df_validation['total_crashes'][0]:,}")
    print(f"🤕 Total Injuries: {df_validation['total_injuries'][0]:,}")
    print(f"💀 Total Deaths: {df_validation['total_deaths'][0]:,}")
    print(f"📊 Total Casualties: {df_validation['total_casualties'][0]:,}")
    
    # Validation against expected ranges
    injuries = df_validation['total_injuries'][0]
    deaths = df_validation['total_deaths'][0]
    
    print("\n" + "=" * 80)
    print("VALIDATION AGAINST NYC OFFICIAL STATISTICS")
    print("=" * 80)
    
    # Deaths validation
    expected_deaths = (3000, 3500)
    if expected_deaths[0] <= deaths <= expected_deaths[1]:
        print(f"✅ Deaths: {deaths:,} is within expected range {expected_deaths[0]:,}-{expected_deaths[1]:,}")
    else:
        print(f"⚠️  Deaths: {deaths:,} is OUTSIDE expected range {expected_deaths[0]:,}-{expected_deaths[1]:,}")
    
    # Injuries validation
    expected_injuries = (500000, 700000)
    if expected_injuries[0] <= injuries <= expected_injuries[1]:
        print(f"✅ Injuries: {injuries:,} is within expected range {expected_injuries[0]:,}-{expected_injuries[1]:,}")
    else:
        print(f"⚠️  Injuries: {injuries:,} is OUTSIDE expected range {expected_injuries[0]:,}-{expected_injuries[1]:,}")
    
    print("\n" + "=" * 80)
else:
    print("⚠️  PERSON_INJURY column not found - validation skipped")

# Close connection
con.close()
print("\n✓ DuckDB connection closed.")

ConnectionException: Connection Error: Connection already closed!